In [1]:
from main import sheet_processor
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os
from utils.processing import DataSampler, create_report, make_json_safe
import traceback

# How to set log level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

## Model Selection

In this section, we specify which models the system will use. To simplify the initial setup and testing workflow, we currently use a single selected model across all processing components: **PII Detection, PII Reflection, Non-PII Detection, and ReadMe Detection**. This approach allows us to validate the end-to-end pipeline without introducing variability from multiple model behaviors.

At the moment, only two models are deployed through the Azure service: **GPT-4.1 Mini** and **GPT-4.1 Nano**. These models are the available choices for experimentation and integration. As the project evolves, additional models can be added or different models can be assigned to specific components for more specialized behavior.


In [9]:
MODEL = 'gpt-5-mini'

ALLOWED_MODELS = ['gpt-4.1-mini', 'gpt-4.1-nano', 'gpt-5-nano', 'gpt-5-mini']

if MODEL not in ALLOWED_MODELS:
    raise ValueError(f'Invalid model: {MODEL}. Please use one of the following: {", ".join(ALLOWED_MODELS)}')

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [10]:
file_path = 'research/data/'  # This is a local test file

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


In [11]:
sampler = DataSampler()

# Ground truth

This part of the code makes the sdd reports, but empty so we can later on fill in the real values


In [12]:
# # Start by making empty reports for each file as groundtruth2 to fill in later

# sampler = DataSampler()

# with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
#     isp = json.load(f)

# isp = isp['default']

# for file in os.listdir('research/data'):
#     file_path = f'research/data/{file}'
#     # If file already exists, skip
#     if os.path.exists(f'research/results/test_results/groundtruth2/{file}.json'):
#         logger.info(f'File {file} already exists. Skipping.')
#         continue
#     try:
#         sdd_report = create_report(file_path)
#         sdd_report = make_json_safe(sdd_report)
#         with open(f'research/results/test_results/groundtruth2/{file}.json', 'w') as f:
#             json.dump(sdd_report, f, indent=4)
#         logger.info(f'Report saved to research/results/test_results/groundtruth2/{file}.json')
#     except Exception as e:
#         logger.warning(f'Error processing file: {file} {e}')
#         logger.info(f'Error details: {traceback.format_exc()}')
#         break
#         continue

# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


In [ ]:
# Start making sdd reports for a model
MODEL = 'gpt-5-mini'
# Start by making empty reports for each file as groundtruth2 to fill in later

# Make a folder for the model
os.makedirs(f'research/results/test_results/{MODEL}', exist_ok=True)

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # Check if file already exists
    output_path = f'research/results/test_results/{MODEL}/{file}.json'
    if os.path.exists(output_path):
        logger.info(f'File {file} already exists, loading existing report')
        # Load the existing report
        with open(output_path, 'r') as f:
            sdd_report = json.load(f)
        logger.info(f'Report loaded from {output_path}')
    else:
        # Check if file is already processed
        if not os.path.exists(f'research/results/test_results/groundtruth2/{file}.json'):
            logger.warning(f'File {file} not found in groundtruth2, skipping')
            continue
        else:
            with open(f'research/results/test_results/groundtruth2/{file}.json', 'r') as f:
                logger.info(f'File {file} found in groundtruth2, loading')
                sdd_report = json.load(f)

    reports = []
    for sheet in sdd_report:
        sdd_report = sheet_processor(sheet, isp, MODEL)
        reports.append(sdd_report)
    with open(output_path, 'w') as f:
        json.dump(reports, f, indent=4)
    logger.info(f'Report saved to {output_path}')

[69417 - 8687722688] 2026-01-15 11:14:20,498 WARNI [__main__:28] File bgd_dataset_joint-msna_refugee_september-2019.xlsx not found in groundtruth2, skipping
[69417 - 8687722688] 2026-01-15 11:14:20,499 INFO  [__main__:20] File afghanistan_june_protection.csv already exists, loading existing report
[69417 - 8687722688] 2026-01-15 11:14:20,500 INFO  [__main__:24] Report loaded from research/results/test_results/gpt-5-nano/afghanistan_june_protection.csv.json
[69417 - 8687722688] 2026-01-15 11:14:20,501 INFO  [main:86] PII classifier model already set to gpt-5-nano. Skipping.
[69417 - 8687722688] 2026-01-15 11:14:20,501 INFO  [main:98] PII reflection model already set to gpt-5-nano. Skipping.
[69417 - 8687722688] 2026-01-15 11:14:20,502 INFO  [main:110] Non-PII model already set to gpt-5-nano. Skipping.
Non-PII classification: {'resource_id': None, 'file_name': 'research/data/afghanistan_june_protection.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2026-01-03 18

Reflecting on PII sensitivity: 100%|██████████| 13/13 [00:04<00:00,  3.24it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/sira_household.xlsx', 'file_url': None, 'sheet_name': 'Sheet1', 'processing_timestamp': '2026-01-03 18:13:48', 'processing_success': True, 'n_records': 999, 'n_columns': 13, 'completion_tokens': 655, 'prompt_tokens': 7938, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'hhID', 'sample_values': ['caap5lrlvgduqk05', 'ckq4r1xlvmbev9y2q', 'ct42s3klvmbaizj1f', 'c608zrtlvmbbtur1r', 'c5lst8nlvmbvgm32'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_orgENUM', 'sample_values': ['FAMOD', 'FAMOD', 'FAMOD', 'FAMOD', 'FAMOD'], 'pii': {'entity_type': 'ORGANIZATION_NAME', 'sensitive': True}}, {'column_name': 'Parent_District', 'sample_values': ['Metuge', 'Metuge', 'Metuge', 'Metuge', 'Metuge'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_Subdistrict', 'sample_values': ['Metuge', 'Metuge', 'Metuge', 'Metuge', 'Metuge'], 'pii': {'entity

Reflecting on PII sensitivity: 100%|██████████| 47/47 [00:11<00:00,  3.96it/s]


[69417 - 8687722688] 2026-01-15 11:15:32,608 INFO  [__main__:41] Report saved to research/results/test_results/gpt-5-nano/afghanistan_june_health.csv.json
[69417 - 8687722688] 2026-01-15 11:15:32,611 WARNI [__main__:28] File isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx not found in groundtruth2, skipping
[69417 - 8687722688] 2026-01-15 11:15:32,612 WARNI [__main__:28] File ACF_Pastoral_Sentinels.xlsx not found in groundtruth2, skipping
[69417 - 8687722688] 2026-01-15 11:15:32,615 WARNI [__main__:28] File acf_donnees_suivi_micro-entreprises_bassikounou_20250129-20250604.xlsx not found in groundtruth2, skipping
[69417 - 8687722688] 2026-01-15 11:15:32,618 INFO  [__main__:32] File ethiopian_health_facilities.csv found in groundtruth2, loading


...

Non-PII classification: {'resource_id': None, 'file_name': 'research/data/afghanistan_june_food.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2026-01-03 18:21:02', 'processing_success': True, 'n_records': 199, 'n_columns': 42, 'completion_tokens': 1391, 'prompt_tokens': 33972, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'observed_time', 'sample_values': ['2025-06-03T07:38:39.507Z', '2025-06-01T13:04:12.381Z', '2025-06-03T10:27:47.558Z', '2025-06-01T13:29:07.920Z', '2025-06-01T13:12:56.969Z'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_version', 'sample_values': ['8', '8', '8', '8', '8'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_name', 'sample_values': ['Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community'], 'pii': {'entity_

Reflecting on PII sensitivity: 100%|██████████| 37/37 [00:07<00:00,  5.28it/s]


[69417 - 8687722688] 2026-01-15 11:25:51,425 ERROR [main:140] Error in README scan classification: 'dict' object has no attribute 'readme' and no __dict__ for setting new attributes
[69417 - 8687722688] 2026-01-15 11:25:51,431 INFO  [__main__:41] Report saved to research/results/test_results/gpt-5-nano/2023-the-effects-on-health-care-of-the-use-of-explosive-weapons-data (1).xlsx.json
[69417 - 8687722688] 2026-01-15 11:25:51,433 INFO  [__main__:32] File afghanistan_june_education.csv found in groundtruth2, loading


Reflecting on PII sensitivity: 100%|██████████| 43/43 [00:09<00:00,  4.57it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/afghanistan_june_education.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2026-01-03 18:21:03', 'processing_success': True, 'n_records': 199, 'n_columns': 43, 'completion_tokens': 1389, 'prompt_tokens': 35855, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'observed_time', 'sample_values': ['2025-06-02T04:49:53.153Z', '2025-06-03T02:18:19.884Z', '2025-06-02T17:29:21.108Z', '2025-06-02T16:06:22.049Z', '2025-06-02T17:12:51.830Z'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_version', 'sample_values': ['10', '10', '10', '10', '10'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_name', 'sample_values': ['Education in Your Community', 'Education in Your Community', 'Education in Your Community', 'Education in Your Community', 'Education in Your Community'], 'pii': {'entity_type': 'None', 'sensitive': Fa

# Save


# Evaluation


In [7]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [8]:
filename = 'data.xlsx'

# Read the groundtruth2 and predictions
with open(f'research/results/test_results/groundtruth2/{filename}.json', 'r') as f:
    groundtruth2 = json.load(f)
with open(f'research/results/test_results/gpt-5-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth2, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth2, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth2, predictions)),
    }
}
metrics

FileNotFoundError: [Errno 2] No such file or directory: 'research/results/test_results/groundtruth2/data.xlsx.json'